# Transient vs quasi-steady PEMFC simulation — comparison with MEA62 data

This notebook replays the MEA62 test-bench polarisation curve (the same log used in
notebook 03) through two simulation approaches:

* **Quasi-steady state (QSS):** `ExplicitSteadyStateModel` called in *vectorised* form
  — all 4 000+ samples in a single call, no Python loop.
* **Transient:** `TransientModel` integrates the coupled MEA-temperature / membrane
  water-content ODEs (Ferrara et al., 2018) under continuously time-varying logged
  conditions, low-pass filtered at 0.1 Hz to remove sensor noise that would otherwise
  force unnecessarily small ODE steps.

Compared quantities: cell voltage, MEA temperature, membrane and CL water contents,
cathode CL liquid saturation, high-frequency resistance (HFR), and CL proton resistance.

**Note:** both models use the same MEA62 parameter set estimated in
`02_parameter_estimation.ipynb`.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.signal import butter, filtfilt

import marapendi as mrpd
from marapendi.cell.transient import TransientModel
from marapendi.cell.explicit_steady_state import ExplicitSteadyStateModel

## 1 — Load and pre-process the MEA62 log

Same CSV parsing as notebook 03, followed by a zero-phase low-pass Butterworth
filter (4th order, cutoff 0.1 Hz) applied to all channel signals.  This removes
sensor noise faster than ~10 s and greatly reduces the number of adaptive ODE
steps needed by the transient solver.

In [ ]:
df = pd.read_csv('monocell_datas_03Mar2026_08h10.csv', sep=';', skiprows=6, encoding='latin1')
df = df.rename(columns=lambda c: c.strip())
old_columns = list(df.columns)
df = df.reset_index()
df.columns = ['Time(s)'] + old_columns[1:] + ['_extra']
df = df.drop(columns=['_extra'])
df['Time(h)'] = (df['Time(s)'] - df['Time(s)'].iloc[0]) / 3600.

CELL_AREA = 25e-4   # m²
N_CELLS   = 1
I_MIN     = 0.5     # A  — exclude near-zero current
T_START_H = 1.6     # h
T_END_H   = 2.8     # h

active = (
    (df['I_Pile(A)'] > I_MIN)
    & (df['Time(h)'] >= T_START_H)
    & (df['Time(h)'] <= T_END_H)
).to_numpy()

df_a     = df[active].copy().reset_index(drop=True)
t_abs_s  = (df_a['Time(h)'] * 3600).to_numpy()
t0_s     = t_abs_s[0]
t_rel_s  = t_abs_s - t0_s   # seconds from window start

# ---- 0.1 Hz low-pass filter ------------------------------------------------
dt_s = float(np.median(np.diff(t_rel_s)))   # median sample interval ≈ 1 s
fs   = 1.0 / dt_s                           # ≈ 1 Hz
fc   = 0.1                                  # cutoff Hz
b, a = butter(4, fc / (fs / 2), btype='low')

def lpf(x):
    return filtfilt(b, a, x)

_I     = np.maximum(lpf(df_a['I_Pile(A)'].to_numpy()), I_MIN)
_T_ca  = lpf((df_a['T_Air_in(°C)'] + 273.15).to_numpy())
_p_ca  = lpf((df_a['P_Air_Out(bara)'] * 1e5).to_numpy())
_xi_ca = np.maximum(lpf(df_a['Stoeckio_air_calc'].to_numpy()), 1.01)
_rh_ca = lpf((df_a['RH_Air_calc(%)'] / 100.).to_numpy())
_T_an  = lpf((df_a['T_H2_In(°C)'] + 273.15).to_numpy())
_p_an  = lpf((df_a['P_h2_out(bara)'] * 1e5).to_numpy())
_xi_an = np.maximum(lpf(df_a['Stoeckio_h2_calc'].to_numpy()), 1.01)
_rh_an = lpf((df_a['RH_h2_calc(%)'] / 100.).to_numpy())
_T_c   = lpf((df_a['T_pile(°C)'] + 273.15).to_numpy())

print(f'Active samples: {active.sum()},  window duration: {t_rel_s[-1]:.0f} s')
print(f'Median sample interval: {dt_s:.2f} s  →  LPF cutoff: {fc} Hz  ({1/fc:.0f} s period)')

## 2 — MEA62 cell assembly

Same 18-parameter estimated configuration as notebook 03.

In [ ]:
fixed_parameters = {
    'radius-carbon': 25e-9, 'ionomer-E-act-cond': 15e6, 'n_s': 2,
    'ionomer-k1': 8.5, 'ionomer-k2': 5.4, 'ionomer-k3': 5.4,
    'gdl-porosity': 0.6, 'pt-wt-percent': 0.4, 'ch-height': 1e-3,
    'gdl-thickness': 150e-6, 'gdl-theta': 120.,
    'gdl-eff-diff-ratio': 0.3, 'cl-abs-perm': 1e-13,
    'wet-transition': 0.4, 'pt-loading': .3e-2, 'ic-ratio': 1.4,
    'ecsa': 60e3, 'memb-thickness': 12e-6,
    'memb-water-diff': 2e-10, 'E-act-memb-diff': 20e6,
    'E-act-memb-abs': 20e6, 'cl-theta': 97.,
    'cl-thermal-cond': 0.22, 'cl-pore-diameter': 40e-9,
}
estimated_parameters = {
    'elec-resistance': 3.2018410582982336e-06,
    'alpha-c': 0.8804552030152384,
    'memb-cond-correction': 10.194306339919532,
    'B_ch': 1.3173241932454605,
    'ionomer-cond-corr': 0.16788866561668214,
    'i0-c': 0.0013603559102389256,
    'memb-cond-exp': 1.6472232706926844,
    'Sh': 0.7956740630180096,
    'E-act-ca': 73404895.12308666,
    'memb-equiv-weight': 707.0461410229138,
    'memb-E-act-cond': 12920411.386859203,
    'gdl-thermal-cond': 0.10151383504290674,
    'gamma-c': 0.7815865333197847,
    'memb-abs-constant': 3.680688030527334e-05,
    'ix-corr': 2.0,
    'ionomer-cond-exp': 1.0,
    'tcr': 0.0009955086394233985,
    'gdl-abs-perm': 9.999999010000095e-12,
}
params = {**fixed_parameters, **estimated_parameters}


def create_cell(params):
    class NewPermModel(mrpd.HydrogenPermeationModel):
        def permeation_flux(self, membrane_thickness, partial_pressure_h2,
                            temperature, pressure_difference, water_vol_fraction):
            return self.permeability_correction_factor * (
                15.7e-9 * np.exp(-20280 / 8.3415 / temperature) +
                water_vol_fraction * 45e-9 * np.exp(-18930 / 8.3145 / temperature)
            ) / 1000 * 100 / 1e5 * partial_pressure_h2 / membrane_thickness

    membrane = mrpd.PFSA(
        ionomer=mrpd.PFSAIonomer(
            equivalent_weight=params['memb-equiv-weight'], dry_density=2000.,
            conductivity_exp=params['memb-cond-exp'],
            conductivity_activation_energy=params['memb-E-act-cond'],
            conductivity_correction=params['memb-cond-correction'],
            reference_water_diffusivity=params['memb-water-diff'],
            reference_water_absorption_coefficient=params['memb-abs-constant'],
            water_diffusivity_activation_energy=params['E-act-memb-diff'],
            water_absorption_activation_energy=params['E-act-memb-abs'],
        ),
        dry_thickness=params['memb-thickness'],
        h2_permeation_model=NewPermModel(permeability_correction_factor=params['ix-corr']),
    )
    orr = mrpd.ElectrochemicalReaction(
        reference_exchange_current_density=params['i0-c'],
        reaction_order=params['gamma-c'], activation_energy=params['E-act-ca'],
        reference_activity=1.01325e5, reference_temperature=353.15,
        number_of_electrons=1, charge_transfer_coeff=params['alpha-c'],
    )
    liq = mrpd.DarcyTransportModel(J_function_exponent=params['wet-transition'])
    gdl = {s: mrpd.GasDiffusionLayer(
        thickness=params['gdl-thickness'], contact_angle=params['gdl-theta'],
        effective_gas_diffusion_ratio=params['gdl-eff-diff-ratio'],
        absolute_permeability=params['gdl-abs-perm'], porosity=params['gdl-porosity'],
        thermal_conductivity=params['gdl-thermal-cond'], two_phase_transport_model=liq,
        transport_resistance_model=mrpd.PorousGasDiffusionModel(
            water_saturation_exponent=params['n_s']),
    ) for s in ['ca', 'an']}
    ch = {s: mrpd.FlowChannel(
        height=params['ch-height'], width=1e-3, n_parallel=1, length=21 * 50e-3,
        reactant='o2' if s == 'ca' else 'h2',
        transport_resistance_model=mrpd.ChannelGasResistanceModel(
            sherwood=params['Sh'], B_ch=params['B_ch']),
    ) for s in ['an', 'ca']}
    ion = mrpd.PFSAIonomer(
        conductivity_correction=params['ionomer-cond-corr'],
        conductivity_exp=params['ionomer-cond-exp'],
        conductivity_activation_energy=params['ionomer-E-act-cond'],
    )
    ca_cl = mrpd.PtCCatalystLayer(
        ecsa=params['ecsa'], platinum_loading=params['pt-loading'], ionomer=ion,
        catalyst_platinum_weight_percent=params['pt-wt-percent'],
        ionomer_to_carbon_ratio=params['ic-ratio'],
        ionomer_k1=params['ionomer-k1'], ionomer_k2=params['ionomer-k2'],
        ionomer_k3=params['ionomer-k3'],
        pore_diameter=params['cl-pore-diameter'], omega_PtO=0,
        carbon_agglomerate_radius=params['radius-carbon'],
        thickness=params['pt-loading'] * 2.8e-6 / 0.1e-2,
        absolute_permeability=params['cl-abs-perm'], contact_angle=params['cl-theta'],
        thermal_conductivity=params['cl-thermal-cond'], reaction=orr,
        two_phase_transport_model=liq,
        transport_resistance_model=mrpd.PorousGasDiffusionModel(water_saturation_exponent=1.5),
    )
    an_cl = mrpd.PtCCatalystLayer(
        ecsa=params['ecsa'], platinum_loading=1e-3,
        catalyst_platinum_weight_percent=params['pt-wt-percent'],
        ionomer_to_carbon_ratio=params['ic-ratio'], ionomer=ion,
        pore_diameter=params['cl-pore-diameter'],
        carbon_agglomerate_radius=params['radius-carbon'],
        thickness=2.8e-6, absolute_permeability=params['cl-abs-perm'],
        contact_angle=params['cl-theta'], thermal_conductivity=params['cl-thermal-cond'],
        two_phase_transport_model=liq,
        transport_resistance_model=mrpd.PorousGasDiffusionModel(water_saturation_exponent=1.5),
    )
    return mrpd.FuelCell(
        electrical_resistance=params['elec-resistance'], area=CELL_AREA,
        ca=mrpd.FuelCellSide(cl=ca_cl, gdl=gdl['ca'], has_gdl=True, has_mpl=False,
                              ch=ch['ca'], thermal_contact_resistance=params['tcr']),
        an=mrpd.FuelCellSide(cl=an_cl, gdl=gdl['an'], has_gdl=True, has_mpl=False,
                              ch=ch['an'], thermal_contact_resistance=params['tcr']),
        membrane=membrane,
    )


cell = create_cell(params)
print('MEA62 cell assembled.')

## 3 — Vectorised quasi-steady simulation

All 4 000+ samples are passed as numpy arrays in a **single** call to
`ExplicitSteadyStateModel.solve`, eliminating Python-loop overhead.
The model already supports array-valued current density; here we also
pass array-valued temperatures, pressures, humidities and stoichiometries
so that each sample's conditions are handled simultaneously.

In [ ]:
ss_model = ExplicitSteadyStateModel()

cond_v = mrpd.CellConditions(
    current_density=_I / CELL_AREA,
    cell_temperature=_T_c,
    ca=mrpd.SideConditions(
        inlet_temperature=_T_ca, outlet_pressure=_p_ca,
        dry_o2_mole_fraction=0.21,
        stoichiometry=_xi_ca, inlet_relative_humidity=_rh_ca,
    ),
    an=mrpd.SideConditions(
        inlet_temperature=_T_an, outlet_pressure=_p_an,
        dry_h2_mole_fraction=1.0,
        stoichiometry=_xi_an, inlet_relative_humidity=_rh_an,
    ),
)

t0_qss = time.perf_counter()
state_v = ss_model.set_initial_conditions(cell, cond_v)
state_v = ss_model.solve(cell, cond_v, state_v)
t_qss   = time.perf_counter() - t0_qss

# Extract QSS diagnostics
qss = {
    'cell_voltage':           np.atleast_1d(state_v.cell_voltage),
    'T_mea':                  np.atleast_1d(state_v.mea_temperature),
    'membrane_water_content': np.atleast_1d(state_v.membrane.water_content),
    'ca_cl_water_content':    np.atleast_1d(state_v.ca.cl.ionomer_water_content),
    'an_cl_water_content':    np.atleast_1d(state_v.an.cl.ionomer_water_content),
    'ca_cl_liquid_saturation':np.atleast_1d(state_v.ca.cl.liquid_saturation),
    'ca_cl_proton_resistance':np.atleast_1d(state_v.ca.cl.proton_resistance),
    'hfr':                    np.atleast_1d(
                                  ss_model.voltage_model.high_frequency_resistance(cell, state_v)
                              ),
}
n_qss = len(_I)
print(f'QSS vectorised: {n_qss} samples in {t_qss:.3f} s  ({1e3*t_qss/n_qss:.3f} ms/sample)')
print(f'Voltage range: {qss["cell_voltage"].min():.3f} – {qss["cell_voltage"].max():.3f} V')

## 4 — Transient simulation

A callable `conditions(t)` interpolates the **filtered** logged signals at
any time *t* (seconds from window start).  The model is initialised at the
SS solution for the first sample, then integrates over the full window.
`solve` computes all diagnostics automatically and stores them in
`sol.diagnostics`.

In [ ]:
def conditions(t):
    """Filtered logged conditions interpolated at time *t* (s from window start)."""
    t = np.clip(float(t), t_rel_s[0], t_rel_s[-1])
    return mrpd.CellConditions(
        current_density=np.atleast_1d(np.interp(t, t_rel_s, _I) / CELL_AREA),
        cell_temperature=float(np.interp(t, t_rel_s, _T_c)),
        ca=mrpd.SideConditions(
            inlet_temperature=float(np.interp(t, t_rel_s, _T_ca)),
            outlet_pressure=float(np.interp(t, t_rel_s, _p_ca)),
            dry_o2_mole_fraction=0.21,
            stoichiometry=float(np.interp(t, t_rel_s, _xi_ca)),
            inlet_relative_humidity=float(np.interp(t, t_rel_s, _rh_ca)),
        ),
        an=mrpd.SideConditions(
            inlet_temperature=float(np.interp(t, t_rel_s, _T_an)),
            outlet_pressure=float(np.interp(t, t_rel_s, _p_an)),
            dry_h2_mole_fraction=1.0,
            stoichiometry=float(np.interp(t, t_rel_s, _xi_an)),
            inlet_relative_humidity=float(np.interp(t, t_rel_s, _rh_an)),
        ),
    )


N_MESH   = 5
tr_model = TransientModel(n_memb_mesh=N_MESH)

_, x0 = tr_model.set_initial_conditions(cell, conditions(0.))
print(f'x0:  T_MEA = {x0[0]:.2f} K,  λ = {np.round(x0[1:], 2)}')

T_SIM = float(t_rel_s[-1])
t0_tr = time.perf_counter()
sol = tr_model.solve(
    cell, conditions,
    t_span=(0., T_SIM),
    x0=x0,
    dense_output=True,
    max_step=30.,
    compute_diagnostics=False,   # we will evaluate at logged sample times below
)
t_tr_solve = time.perf_counter() - t0_tr
print(f'ODE solve: status={sol.status}, steps={len(sol.t)}, time={t_tr_solve:.2f} s')

## 5 — Evaluate transient diagnostics at logged sample times

`TransientModel.evaluate` runs the full physics pipeline at each requested
time point and returns a dict of arrays matching the QSS diagnostics.

In [ ]:
t0_tr_post = time.perf_counter()
tr = tr_model.evaluate(cell, conditions, t_rel_s, x_eval=sol.sol(t_rel_s))
t_tr_post = time.perf_counter() - t0_tr_post
t_tr_total = t_tr_solve + t_tr_post

print(f'Evaluation: {len(t_rel_s)} samples in {t_tr_post:.2f} s  ({1e3*t_tr_post/n_qss:.2f} ms/sample)')
print(f'Transient total: {t_tr_total:.2f} s')
print(f'Voltage range: {tr["cell_voltage"].min():.3f} – {tr["cell_voltage"].max():.3f} V')

## 6 — Computational cost

In [ ]:
print('─' * 56)
print(f'{"Method":<30} {"Time (s)":>10}  {"ms/sample":>12}')
print('─' * 56)
print(f'{"QSS vectorised":<30} {t_qss:>10.3f}  {1e3*t_qss/n_qss:>12.3f}')
print(f'{"Transient ODE solve":<30} {t_tr_solve:>10.2f}  {"–":>12}')
print(f'{"Transient evaluate":<30} {t_tr_post:>10.2f}  {1e3*t_tr_post/n_qss:>12.2f}')
print(f'{"Transient total":<30} {t_tr_total:>10.2f}  {1e3*t_tr_total/n_qss:>12.2f}')
print('─' * 56)
print(f'Transient / QSS ratio: {t_tr_total/t_qss:.1f}×')
print(f'ODE steps: {len(sol.t)}  '
      f'(simulated {T_SIM:.0f} s,  avg step {T_SIM/len(sol.t):.1f} s)')

## 7 — Comparison plots

Shared time axis: hours in the polarisation-curve window.

In [ ]:
t_h    = T_START_H + t_rel_s / 3600.   # hours
t_all  = df['Time(h)'].to_numpy()
window = (t_all >= T_START_H) & (t_all <= T_END_H)
t_w    = t_all[window]
meas_V = df['U_Pile(V)'].to_numpy()[window] / N_CELLS
meas_I = df['I_Pile(A)'].to_numpy()[window] / CELL_AREA * 1e-4   # A/cm²

### 7.1 Cell voltage

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

axes[0].plot(t_w, meas_I, 'k', lw=0.8)
axes[0].set_ylabel('Current density (A/cm²)')
axes[0].grid(True, alpha=0.4)

axes[1].plot(t_w, meas_V, 'k', lw=1.2, label='Measured')
axes[1].plot(t_h, qss['cell_voltage'], 'C0', lw=1.2, label='QSS')
axes[1].plot(t_h, tr['cell_voltage'],  'C1', lw=1.2, alpha=0.85, label='Transient')
axes[1].set_ylabel('Cell voltage (V)')
axes[1].legend()
axes[1].grid(True, alpha=0.4)

axes[2].plot(t_h, 1e3 * (meas_V - qss['cell_voltage']), 'C0', lw=1., label='V_meas − V_QSS')
axes[2].plot(t_h, 1e3 * (meas_V - tr['cell_voltage']),  'C1', lw=1., label='V_meas − V_tr')
axes[2].axhline(0, color='k', lw=0.7)
axes[2].set_ylabel('Voltage error (mV)')
axes[2].set_xlabel('Time (h)')
axes[2].legend()
axes[2].grid(True, alpha=0.4)

fig.suptitle('MEA62 — cell voltage')
fig.tight_layout()
plt.show()

### 7.2 MEA temperature

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t_h, qss['T_mea'] - 273.15,  'C0', lw=1.2, label='QSS T_MEA')
ax.plot(t_h, tr['T_mea'] - 273.15,   'C1', lw=1.2, label='Transient T_MEA')
ax.plot(t_w, df['T_pile(°C)'].to_numpy()[window], 'k--', lw=0.9, label='Logged T_stack')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Temperature (°C)')
ax.set_title('MEA temperature — QSS vs transient')
ax.legend()
ax.grid(True, alpha=0.4)
fig.tight_layout()
plt.show()

### 7.3 Membrane and CL water contents

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

ax = axes[0]
ax.plot(t_h, qss['membrane_water_content'], 'C0', lw=1.2, label='QSS')
ax.plot(t_h, tr['membrane_water_content'],  'C1', lw=1.2, label='Transient')
ax.set_ylabel('λ_membrane (mol/mol)')
ax.set_title('Mean membrane water content')
ax.legend()
ax.grid(True, alpha=0.4)

ax = axes[1]
ax.plot(t_h, qss['ca_cl_water_content'], 'C0', lw=1.2, label='QSS')
ax.plot(t_h, tr['ca_cl_water_content'],  'C1', lw=1.2, label='Transient')
ax.set_ylabel('λ_CL cathode (mol/mol)')
ax.set_title('Cathode CL ionomer water content')
ax.legend()
ax.grid(True, alpha=0.4)

ax = axes[2]
ax.plot(t_h, qss['an_cl_water_content'], 'C0', lw=1.2, label='QSS')
ax.plot(t_h, tr['an_cl_water_content'],  'C1', lw=1.2, label='Transient')
ax.set_ylabel('λ_CL anode (mol/mol)')
ax.set_title('Anode CL ionomer water content')
ax.set_xlabel('Time (h)')
ax.legend()
ax.grid(True, alpha=0.4)

fig.suptitle('MEA62 — water contents')
fig.tight_layout()
plt.show()

### 7.4 Cathode CL liquid water saturation

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t_h, qss['ca_cl_liquid_saturation'], 'C0', lw=1.2, label='QSS')
ax.plot(t_h, tr['ca_cl_liquid_saturation'],  'C1', lw=1.2, label='Transient')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Liquid saturation (–)')
ax.set_title('Cathode CL liquid water saturation')
ax.legend()
ax.grid(True, alpha=0.4)
fig.tight_layout()
plt.show()

### 7.5 HFR and cathode CL proton resistance

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax = axes[0]
ax.plot(t_h, qss['hfr'] * 1e4, 'C0', lw=1.2, label='QSS')
ax.plot(t_h, tr['hfr'] * 1e4,  'C1', lw=1.2, label='Transient')
ax.set_ylabel('HFR (mΩ·cm²)')
ax.set_title('High-frequency resistance (membrane + contact)')
ax.legend()
ax.grid(True, alpha=0.4)

ax = axes[1]
ax.plot(t_h, qss['ca_cl_proton_resistance'] * 1e4, 'C0', lw=1.2, label='QSS')
ax.plot(t_h, tr['ca_cl_proton_resistance'] * 1e4,  'C1', lw=1.2, label='Transient')
ax.set_ylabel('R_CL (mΩ·cm²)')
ax.set_title('Cathode CL proton resistance')
ax.set_xlabel('Time (h)')
ax.legend()
ax.grid(True, alpha=0.4)

fig.suptitle('MEA62 — resistances')
fig.tight_layout()
plt.show()

### 7.6 Membrane water-content profile evolution (transient)

In [ ]:
xi = np.linspace(0, 1, N_MESH)

n_snaps   = 8
t_snaps_s = np.linspace(t_rel_s[0], t_rel_s[-1], n_snaps)
x_snaps   = sol.sol(t_snaps_s)
lmbd_snaps = x_snaps[1:, :]
t_snaps_h = T_START_H + t_snaps_s / 3600.

cmap_snap = cm.coolwarm
colors    = cmap_snap(np.linspace(0, 1, n_snaps))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
for k in range(n_snaps):
    ax.plot(xi, lmbd_snaps[:, k], 'o-', ms=4, color=colors[k],
            label=f'{t_snaps_h[k]:.2f} h')
ax.set_xlabel('ξ (anode → cathode)')
ax.set_ylabel('λ (mol H₂O / mol SO₃⁻)')
ax.set_title('Profile snapshots')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.4)

ax = axes[1]
t_d   = np.linspace(t_rel_s[0], t_rel_s[-1], 300)
x_d   = sol.sol(t_d)
t_d_h = T_START_H + t_d / 3600.
cmap_n = cm.viridis
nc     = cmap_n(np.linspace(0, 1, N_MESH))
for k in range(N_MESH):
    ax.plot(t_d_h, x_d[1 + k], color=nc[k], label=f'ξ={xi[k]:.2f}')
ax.set_xlabel('Time (h)')
ax.set_ylabel('λ (mol H₂O / mol SO₃⁻)')
ax.set_title('Node time evolution')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.4)

fig.suptitle('MEA62 — membrane water-content profile (transient)')
fig.tight_layout()
plt.show()